# Qwen3.8-27B Abliterated on Colab T4 (AirLLM)

> Colab for Google AI Pro (CCU) + AirLLM layer streaming. 27B full model,
> tanpa quantization, tanpa pruning. Muat dalam T4 16GB.

**Sebelum run:**
1. **Runtime -> Change runtime type -> Hardware accelerator: GPU (T4)**
2. Sistem memory profile: pilih **High Memory** (24GB) kalau ada option - lebih safe
3. Run cells bawah ikut urutan. **Option A (default) atau Option B (4bit)** - pilih SATU je untuk jimat disk
4. First run download ~54GB weights - kena sabar (speed depend network Colab)

**Apa yang jadi:** AirLLM split model jadi layer shards (disk), GPU cuma pegang
1 layer setiap forward pass. GPU+RAM+disk semua kerja - memang tu setting
"CPU + RAM + GPU" yang dimaksud.

**Expectation (anggaran, kena test):** bf16 ~1-3 tok/s, 4bit ~3-5 tok/s.
Jawapan kat bawah dari actual benchmark, bukan marketing.


In [ ]:
!pip install -U airllm bitsandbytes transformers
!nvidia-smi   # confirm T4 ada
!df -h /       # check disk (kena ada ~60GB free utk bf16, ~35GB utk 4bit)
!free -h      # check system RAM


## Kenapa T4 cukup (GPU + CPU + RAM + disk)

| Komponen | Role dalam AirLLM |
|---|---|
| T4 16GB VRAM | Pegang 1 layer aktif + KV cache. Qwen3.8-27B perlukan ~3.33GB sahaja -> lega |
| System RAM | Python, tokenizer, buffer. High memory profile (24GB) lebih selesa |
| Disk Colab | Simpan layer shards (split model). bf16 ~54GB; guna `delete_original=True` bila dah split |
| CPU | Orkestrasi, decode. Bukan bottleneck utama |

Bottleneck sebenar = disk read speed Colab. Tu kenapa kita benchmark tok/s
di bawah - angka real, bukan anggapan.


In [ ]:
import os, shutil, torch
print("CUDA:", torch.cuda.is_available(), "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("VRAM free/total:", torch.cuda.get_device_properties(0))
total, used, free = shutil.disk_usage("/")
print(f"Disk / : {free/1e9:.0f}GB free / {total/1e9:.0f}GB total")
print("HF cache:", os.path.expanduser("~/.cache/huggingface"))


## Option A - Default (bf16, quality penuh)

Download ~54GB + split ~54GB (lepas `delete_original=True`, jimat half).
Kualiti paling tinggi, speed paling slow. Pilih A kalau nak quality.


In [ ]:
from airllm import AutoModel
import time

# First run: download + split (disk-heavy, sabar).
# delete_original=True buang weights asal lepas split - jimat disk.
model = AutoModel.from_pretrained(
    "huihui-ai/Huihui-Qwen3.8-27B-abliterated",
    delete_original=True,
    profiling_mode=True,
)
print("Loaded:", type(model).__name__)

PROMPT = "Explain dalam 2 ayat apa Qwen3.8-27B abliterated ni."
msgs = [{"role": "user", "content": PROMPT}]
text = model.tokenizer.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
toks = model.tokenizer(text, return_tensors="pt", truncation=True, max_length=512, padding=False)

t0 = time.time()
out = model.generate(toks["input_ids"].cuda(), max_new_tokens=96, use_cache=True, return_dict_in_generate=True)
dt = time.time() - t0
seq = out.sequences[0]
n = seq.shape[0] - toks["input_ids"].shape[0]
print(f"{n} tokens dalam {dt:.1f}s -> {n/dt:.2f} tok/s")
print("=== OUTPUT ===")
print(model.tokenizer.decode(seq, skip_special_tokens=True))


## Option B - 4bit block compression (laju ~3x, quality sikit turun)

Guna `bitsandbytes` block-wise quantization - laju 3x ikut claim rasmi AirLLM,
accuracy loss kecil. Disk lebih jimat (~27GB). Pilih B kalau nak speed.
(JANGAN run A + B sekali dalam same runtime - disk penuh)


In [ ]:
from airllm import AutoModel
import time

model = AutoModel.from_pretrained(
    "huihui-ai/Huihui-Qwen3.8-27B-abliterated",
    compression='4bit',
    delete_original=True,
    profiling_mode=True,
)
print("Loaded (4bit):", type(model).__name__)

PROMPT = "Explain dalam 2 ayat apa Qwen3.8-27B abliterated ni."
msgs = [{"role": "user", "content": PROMPT}]
text = model.tokenizer.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
toks = model.tokenizer(text, return_tensors="pt", truncation=True, max_length=512, padding=False)

t0 = time.time()
out = model.generate(toks["input_ids"].cuda(), max_new_tokens=96, use_cache=True, return_dict_in_generate=True)
dt = time.time() - t0
seq = out.sequences[0]
n = seq.shape[0] - toks["input_ids"].shape[0]
print(f"{n} tokens dalam {dt:.1f}s -> {n/dt:.2f} tok/s (4bit)")
print("=== OUTPUT ===")
print(model.tokenizer.decode(seq, skip_special_tokens=True))


## Chat loop

Model dah load kat Option A/B? Guna cell ni. Colab support `input()` -
tepa prompt, Enter, baca jawapan. `!` command shell, `.ipynb` keluar loop.


In [ ]:
def ask(q, max_new=160):
    msgs = [{"role": "user", "content": q}]
    text = model.tokenizer.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    toks = model.tokenizer(text, return_tensors="pt", truncation=True, max_length=512, padding=False)
    t0 = time.time()
    out = model.generate(toks["input_ids"].cuda(), max_new_tokens=max_new, use_cache=True, return_dict_in_generate=True)
    dt = time.time() - t0
    seq = out.sequences[0]
    ans = model.tokenizer.decode(seq, skip_special_tokens=True)
    n = seq.shape[0] - toks["input_ids"].shape[0]
    print(f"({n/max(0.1,dt):.1f} tok/s)")
    print(ans)

print("Mode chat. Tekan soalan + Enter. Taip 'exit' untuk keluar.")
while True:
    q = input("b> ")
    if q.strip().lower() in ("exit", "quit", "keluar"):
        break
    if not q.strip():
        continue
    ask(q)


## Catatan penting (jujur)

- **ToS Colab**: web service / SSH / API server dari Colab runtime = **dilarang**.
  Notebook peribadi je. Nak serve API keluar, download model ke VPS (Oracle A1 ada 24GB RAM
  tapi slow ~1 tok/s; GPU A100 40GB = jalan laju).
- 27B ni model **abliterated/uncensored** - output mungkin kasar, b dah aware.
- Bila CCU Jio/Google b dah roll: T4 tu jadi bukan lagi limit - boleh upgrade
  ke L4/A100 dan speed naik beberapa x tanpa ubah notebook ni.
